# Week 7 — Request a structured synthetic response

**Research task:** Predict one held-out `POLVIEWS` answer for a deidentified teaching profile, then separate schema validity from predictive evidence.

**Python introduced:** JSON Schema, required fields, numeric lists, `len(...)`, `sum(...)` and Boolean checks.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session07/session07_synthetic_representation.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.


In [ ]:
SESSION = "session07"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib.util as setup_importlib
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath("/content/GenAI_Soc2026")
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The course repository root could not be found. Start Jupyter from the "
            "GenAI_Soc2026 folder with: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Choose a route and store one deidentified profile and survey item

`profile` is a dictionary containing public-use teaching attributes; `survey_item` is the exact question string. `held_out_human_answer` is stored separately and is not included in the prompt. This separation prevents the model from seeing the answer it is meant to predict.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.


In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
profile = {
    "age_group": "30–44",
    "education": "bachelor's degree",
    "region": "South",
    "party_identification": "independent",
}
question = "On a scale from 1 (very liberal) to 7 (very conservative), where would you place yourself?"
held_out_human_answer = 4
print(profile)
print(question)


## Define the exact output structure

The schema requires an integer category from 1 to 7 and a seven-number probability list. `required` says both fields must appear; `additionalProperties: False` prohibits extra fields. These rules concern readable structure and numeric bounds, not whether the prediction represents people well.


In [ ]:
schema = {
    "type": "object",
    "properties": {
        "predicted_category": {"type": "integer", "minimum": 1, "maximum": 7},
        "probabilities": {
            "type": "array", "items": {"type": "number"},
            "minItems": 7, "maxItems": 7,
        },
    },
    "required": ["predicted_category", "probabilities"],
    "additionalProperties": False,
}
prompt = (
    "Predict this deidentified respondent's answer. Return seven probabilities in "
    "order from 1 to 7. Profile: " + json.dumps(profile) + " Item: " + question
)
messages = [{"role": "user", "content": prompt}]

## Make the selected structured-output call

The prompt joins the profile and exact item into one message. OpenRouter places the schema in `response_format`; Ollama receives it through `format`. Both branches preserve the returned JSON text in `raw_json` before any field is extracted.


In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL, messages=messages, temperature=0,
            response_format={"type": "json_schema", "json_schema": {
                "name": "polviews_prediction", "strict": True, "schema": schema,
            }},
        )
    raw_output = response.choices[0].message.content
else:
    response = ollama.chat(
        model=LOCAL_MODEL, messages=messages, format=schema,
        options={"temperature": 0},
    )
    raw_output = response.message.content
print("Raw JSON text:", raw_output)

## Parse, check structure and only then compare with the held-out answer

Parsing creates a dictionary. `len(probabilities)` counts entries and `sum(probabilities)` adds them; comparisons produce Booleans checking seven values, an approximate total of one and values between zero and one. Only after those mechanical checks is `predicted_category` compared with the held-out human answer. One match is not population validation.


In [ ]:
prediction = json.loads(raw_output)
probabilities = prediction["probabilities"]
seven_values = len(probabilities) == 7
sum_is_close = abs(sum(probabilities) - 1.0) < 0.02
matches_human = prediction["predicted_category"] == held_out_human_answer
print("Predicted category:", prediction["predicted_category"])
print("Seven probabilities:", seven_values)
print("Sum close to one:", sum_is_close)
print("Held-out human answer:", held_out_human_answer)
print("Exact match:", matches_human)

# ONE CHANGE: change age_group to "18–29" and rerun.

## Methodological check

A valid schema means Python can retrieve the required fields. One exact match does not establish individual accuracy, group fidelity or population representation.
## Completion recording

Use one route, run the original profile, change only `age_group` and rerun. Explain every schema field, the raw JSON, the parsed list and all three checks. State why neither prediction establishes representativeness.

Explain every input and output aloud. Never show the shared key.